# 02 - Forecasting Experiments

## Related work and model selection

Cellular traffic forecasting has been approached with three broad families of
methods, each represented by one of the three models built in this notebook.

**Statistical/seasonal models.** Ferreira et al. [3] survey and benchmark
ARMA/ARIMA/SARIMA alongside neural approaches for network traffic forecasting
and note that seasonal ARIMA variants suit traffic with strong, regular
periodicity but become costly to fit as the seasonal period grows. Azari et
al. [2] compare ARIMA directly against LSTM on cellular traffic and find ARIMA
competitive on short, regular series, with LSTM's advantage growing with more
training data and finer granularity - motivating including *both* a
statistical and a deep-learning model here, not just one.

**Tree-based / feature-engineered ML models.** Kim [4] applies gradient
boosting ensembles to network traffic prediction and reports competitive
accuracy at a fraction of the training cost of deep sequence models, at the
expense of requiring the modeler to hand-engineer the lag structure rather
than letting the model discover it.

**Deep sequential models.** Santos et al. [1] train LSTM and GRU models on
*this same Milan dataset* for short-term mobile Internet traffic prediction
and report that recurrent models capture the daily/weekly structure well but
degrade during atypical periods not well represented in training data - a
finding directly relevant to the failure-case analysis in
`03_model_comparison.ipynb`.

**How `01_eda.ipynb`'s findings informed model selection.** The EDA showed the
traffic series is strongly daily-periodic (the hour x weekday heatmap), has
autocorrelation that decays in a damped-periodic pattern over several days
while its partial autocorrelation is concentrated in the first few lags (the
ACF/PACF plots), and is stationary in levels (ADF test). That combination -
strong seasonality, short-range direct dependence, and literature evidence
that no single paradigm dominates - motivated three *structurally different*
models rather than three variants of one architecture:

1. **`SARIMAForecaster`** - a statistical model representing the daily
   seasonality explicitly (Fourier terms, motivated by the ACF's 144-lag
   periodicity and the heatmap's weekday/weekend contrast).
2. **`GBMForecaster`** - a tree-based model consuming the short-range
   dependence (PACF-motivated lags) and seasonality as engineered features.
3. **`LSTMForecaster`** - a recurrent model that learns temporal structure
   end-to-end from a raw window of history, rather than from hand-picked
   features.

### References
[1] G. L. Santos, P. Rosati, T. Lynn, J. Kelner, D. Sadok, and P. T. Endo,
"Predicting short-term mobile Internet traffic from Internet activity using
recurrent neural networks," *Int. J. Netw. Manag.*, vol. 32, no. 3, e2191,
2022.
[2] A. Azari, P. Papapetrou, S. Denic, and G. Peters, "Cellular traffic
prediction and classification: a comparative evaluation of LSTM and ARIMA,"
in *Discovery Science (DS 2019)*, LNCS vol. 11828, Springer, Cham, 2019,
pp. 129-144.
[3] G. O. Ferreira, C. Ravazzi, F. Dabbene, G. Calafiore, and M. Fiore,
"Forecasting network traffic: a survey and tutorial with open-source
comparative evaluation," *IEEE Access*, vol. 11, 2023.
[4] H. Kim, "Network traffic prediction using gradient boosting ensemble
method," in *Proc. 2024 7th Artificial Intelligence and Cloud Computing Conf.
(AICCC)*, ACM, 2025, pp. 608-614.
[5] R. J. Hyndman and G. Athanasopoulos, *Forecasting: Principles and
Practice*, 3rd ed., OTexts, 2021, ch. 12 (Fourier terms for long seasonal
periods - see `forecasting/sarima_forecaster.py`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import SquareSeries, TUNING_TRAIN_END, TUNING_VAL_END, FINAL_TRAIN_END, TEST_START, TEST_END
from forecasting.sarima_forecaster import SARIMAForecaster
from forecasting.gbm_forecaster import GBMForecaster
from forecasting.lstm_forecaster import LSTMForecaster
from forecasting.evaluation import WalkForwardEvaluator
from forecasting.search import HyperparameterSearch
from forecasting.tracking import ExperimentTracker

COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

with open(ROOT / "results" / "top_squares.json") as f:
    top_info = json.load(f)
TOP3 = top_info["top3_square_ids"]
TOP_SQUARE = TOP3[0]
print("Top-3 squares:", TOP3, "| tuning/search runs only on:", TOP_SQUARE)

tracker = ExperimentTracker(RESULTS_DIR / "experiment_log.csv")
evaluator = WalkForwardEvaluator()

Top-3 squares: [5161, 5059, 5259] | tuning/search runs only on: 5161


## Evaluation protocol

Every model is scored identically, one-step-ahead, via `WalkForwardEvaluator`:
at each target timestamp, the model only ever conditions on the *true* series
strictly before that point, never on its own prior prediction.

Data is split by time (10-minute resolution):

| Split | Range | Purpose |
|---|---|---|
| `tuning_train` | Nov 1 - Dec 8 | fit candidates during hyperparameter search |
| `tuning_val` | Dec 9 - Dec 15 (1 week) | select hyperparameters by validation RMSE |
| `final_train` | Nov 1 - Dec 15 | fit the chosen-hyperparameter model per square |
| `test` | **Dec 16 - Dec 22** (1 week) | held out; used for all reported plots/tables |

Hyperparameter search (`HyperparameterSearch`) runs **once, on the
highest-traffic square only**, and the winning configuration is reused on the
other two top squares - a deliberate compute/rigor trade-off: exhaustively
re-tuning per square would triple the search cost, and the three top squares
share the same underlying daily/weekly seasonal structure (`01_eda.ipynb`), so
hyperparameters selected for temporal structure should transfer reasonably
well even though absolute traffic levels differ. Every trial - and the
closing rationale for the winning configuration - is logged by
`ExperimentTracker` to `results/experiment_log.csv`, not printed and
discarded.

In [2]:
top_series = SquareSeries(TOP_SQUARE, COMBINED_PATH).load()

sarima_search = HyperparameterSearch(SARIMAForecaster, tracker)
t0 = time.time()
sarima_result = sarima_search.run(top_series, TUNING_TRAIN_END, top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index, square_id=TOP_SQUARE, model_name="SARIMA")
print(f"SARIMA search: {time.time()-t0:.1f}s")
print(sarima_result["rationale"])
pd.DataFrame(sarima_result["all_results"])

SARIMA search: 595.2s
Selected {'order': (2, 1, 2), 'n_harmonics': 2} - lowest validation RMSE (202.17) among 6 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'order': (1, 0, 1), 'n_harmonics': 2}",1.876268,138.424597,12.006434,215.305087
1,"{'order': (1, 0, 1), 'n_harmonics': 3}",2.527881,138.727352,12.037087,215.639748
2,"{'order': (2, 0, 1), 'n_harmonics': 2}",4.495791,139.156642,12.067098,215.992579
3,"{'order': (2, 0, 1), 'n_harmonics': 3}",8.785239,139.526678,12.103007,216.394377
4,"{'order': (2, 1, 2), 'n_harmonics': 2}",3.100052,138.662594,10.944156,202.170387
5,"{'order': (2, 1, 2), 'n_harmonics': 3}",5.325259,139.276841,10.990360,202.976801


**Interpretation.** The table above shows every `(order, n_harmonics)`
combination tried and its validation RMSE. The rationale line states which
one won and by how much. Differencing and harmonic count matter here because
they directly encode how much of the daily/weekly structure found in
`01_eda.ipynb` the model gets to use explicitly versus having to approximate
through the ARIMA error process alone.

In [3]:
gbm_search = HyperparameterSearch(GBMForecaster, tracker)
val_index = top_series.loc[TUNING_TRAIN_END + pd.Timedelta(minutes=10):TUNING_VAL_END].index
t0 = time.time()
gbm_result = gbm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="GBM")
print(f"GBM search: {time.time()-t0:.1f}s")
print(gbm_result["rationale"])
pd.DataFrame(gbm_result["all_results"])

GBM search: 69.1s
Selected {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1} - lowest validation RMSE (158.28) among 4 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'n_estimators': 100, 'max_depth': 3, 'learnin...",3.560420,101.861326,7.690756,158.280875
1,"{'n_estimators': 200, 'max_depth': 3, 'learnin...",6.167706,102.870391,7.662137,160.143524
2,"{'n_estimators': 200, 'max_depth': 4, 'learnin...",8.131576,101.366655,7.559657,159.347097
3,"{'n_estimators': 300, 'max_depth': 4, 'learnin...",12.652518,102.138056,7.609234,160.546820


**Interpretation.** Gradient boosting's grid varies ensemble size, tree depth,
and learning rate. If the smallest candidate in the grid wins, that's a sign
the engineered lag/calendar feature set (Section 5 of `forecasting/
gbm_forecaster.py`, chosen from the EDA's PACF result) is already informative
enough that extra trees or depth mainly add variance rather than reducing
bias - worth checking against the table above.

In [4]:
lstm_search = HyperparameterSearch(LSTMForecaster, tracker)
t0 = time.time()
lstm_result = lstm_search.run(top_series, TUNING_TRAIN_END, val_index, square_id=TOP_SQUARE, model_name="LSTM")
print(f"LSTM search: {time.time()-t0:.1f}s")
print(lstm_result["rationale"])
pd.DataFrame(lstm_result["all_results"])

LSTM search: 140.9s
Selected {'hidden_size': 32, 'num_layers': 1, 'lr': 0.001} - lowest validation RMSE (172.43) among 3 candidates.


,params,train_seconds,mae,mape,rmse
0,"{'hidden_size': 16, 'num_layers': 1, 'lr': 0.001}",25.145334,123.913671,12.042414,178.936431
1,"{'hidden_size': 32, 'num_layers': 1, 'lr': 0.001}",40.697238,121.913130,12.699054,172.429073
2,"{'hidden_size': 32, 'num_layers': 2, 'lr': 0.0...",66.961260,131.698262,12.681236,188.282308


**Interpretation.** The LSTM grid varies hidden size, layer count, and
learning rate, kept deliberately small (see `forecasting/lstm_forecaster.py`
docstring) given this is a CPU-only, laptop-class run - training time for
each candidate is visible in the table above alongside its validation error,
making the accuracy/compute trade-off explicit rather than assumed.

In [5]:
# Final fit + one-step-ahead evaluation (Dec 16-22), all 3 top squares.
best_hp = {
    "SARIMA": sarima_result["best_params"],
    "GBM": gbm_result["best_params"],
    "LSTM": lstm_result["best_params"],
}
forecaster_classes = {"SARIMA": SARIMAForecaster, "GBM": GBMForecaster, "LSTM": LSTMForecaster}

timing_rows = []
for square_id in TOP3:
    series = SquareSeries(square_id, COMBINED_PATH).load()
    final_train = series.loc[:FINAL_TRAIN_END]
    test_index = series.loc[TEST_START:TEST_END].index

    for model_name, cls in forecaster_classes.items():
        model = cls(**best_hp[model_name])
        t0 = time.perf_counter()
        model.fit(final_train)
        train_seconds = time.perf_counter() - t0

        result = evaluator.run(model, series, test_index)
        metrics = {"mae": result["mae"], "mape": result["mape"], "rmse": result["rmse"]}

        tracker.log(
            model=model_name, params=best_hp[model_name], metrics=metrics,
            rationale=f"Final Dec16-22 evaluation on square {square_id}",
            square_id=square_id, phase="final",
            train_seconds=train_seconds, predict_seconds=result["predict_seconds"],
        )
        timing_rows.append({
            "square": square_id, "model": model_name,
            "train_seconds": round(train_seconds, 3),
            "predict_seconds": round(result["predict_seconds"], 3),
            **metrics,
        })

        result["predictions"].to_csv(RESULTS_DIR / f"predictions_{model_name.lower()}_{square_id}.csv", header=["prediction"])
        result["actuals"].to_csv(RESULTS_DIR / f"actuals_{square_id}.csv", header=["actual"])

        print(f"square={square_id} model={model_name:7s} MAE={metrics['mae']:.2f} MAPE={metrics['mape']:.2f} RMSE={metrics['rmse']:.2f} "
              f"train={train_seconds:.1f}s predict={result['predict_seconds']:.1f}s")

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(RESULTS_DIR / "timing.csv", index=False)
timing_df

square=5161 model=SARIMA  MAE=118.15 MAPE=10.94 RMSE=169.08 train=3.5s predict=112.5s


square=5161 model=GBM     MAE=80.67 MAPE=8.64 RMSE=117.48 train=3.4s predict=9.1s


square=5161 model=LSTM    MAE=91.46 MAPE=9.72 RMSE=136.58 train=41.8s predict=2.1s


square=5059 model=SARIMA  MAE=109.63 MAPE=10.53 RMSE=146.80 train=6.9s predict=108.5s


square=5059 model=GBM     MAE=69.73 MAPE=7.33 RMSE=99.16 train=3.6s predict=9.5s


square=5059 model=LSTM    MAE=75.93 MAPE=8.48 RMSE=105.26 train=42.3s predict=2.4s


square=5259 model=SARIMA  MAE=71.30 MAPE=7.57 RMSE=103.18 train=12.1s predict=110.1s


square=5259 model=GBM     MAE=64.39 MAPE=7.23 RMSE=90.63 train=3.4s predict=9.4s


square=5259 model=LSTM    MAE=76.01 MAPE=9.05 RMSE=107.12 train=41.1s predict=2.2s


,square,model,train_seconds,predict_seconds,mae,mape,rmse
0,5161,SARIMA,3.464,112.546,118.148081,10.941124,169.075760
1,5161,GBM,3.436,9.093,80.668467,8.639132,117.483657
2,5161,LSTM,41.796,2.145,91.459941,9.720637,136.583626
3,5059,SARIMA,6.915,108.516,109.629011,10.526495,146.797687
4,5059,GBM,3.559,9.505,69.728171,7.330903,99.161200
5,5059,LSTM,42.279,2.445,75.931211,8.476002,105.257291
6,5259,SARIMA,12.141,110.109,71.304030,7.572055,103.180553
7,5259,GBM,3.385,9.417,64.393131,7.230160,90.625046
8,5259,LSTM,41.114,2.165,76.006685,9.045989,107.121330


**Interpretation.** This table is the raw evidence `03_model_comparison.ipynb`
builds its plots, per-square tables, and comparative discussion from - nothing
downstream recomputes predictions or metrics, it only reads what was written
to `results/` here. At a glance: compare MAE/RMSE across models within a
square (which model wins, and by how much) and across squares for the same
model (does the ranking hold as traffic characteristics change) - both
questions the final report needs answered with evidence, not just the
single lowest number.

`results/experiment_log.csv` now contains every tuning trial plus these final
per-square runs, each tagged by `phase` (`tuning`, `selected`, `final`) so
`03_model_comparison.ipynb` can filter it without re-deriving anything.